In [5]:
from pathlib import Path
import sys

%load_ext autoreload
%autoreload 2

dirs = Path().resolve().parent

if dir not in sys.path:
    sys.path.append(str(dirs))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
from utils.garch import simulate_garch_from_windows
import joblib
import yfinance as yf
import math
from utils.utils import log_transform, split_into_windows
import numpy as np

In [17]:
ticker = "^GSPC"
start_interval = "2010-01-01"
end_interval = "2026-01-01"
interval = "1d"

raw_snp500 = yf.Ticker(ticker).history(
    start=start_interval,
    end=end_interval,
    interval=interval
)["Close"].to_numpy()

n = len(raw_snp500)
split = math.ceil(n * 0.2)

train_end = n - 2 * split

train_raw_snp500 = raw_snp500[:train_end]

train_snp500 = log_transform(train_raw_snp500)

train_snp500 = train_snp500[~np.isnan(train_snp500)]

train_windows_128 = split_into_windows(train_snp500, 128)
train_windows_256 = split_into_windows(train_snp500, 256)
train_windows_512 = split_into_windows(train_snp500, 512)

In [20]:
train_windows_128.shape, train_windows_256.shape, train_windows_512.shape

((18, 128), (9, 256), (4, 512))

In [21]:
config = {
  "mean": "zero",
  "p": 1,
  "q": 1,
  "burn": 500
}

In [22]:
sim_windows_128 = np.array(
    simulate_garch_from_windows(train_windows_128, **config)
)

sim_data_128 = {
    "window_size": 128,
    "config": config.copy(),
    "sim_data": sim_windows_128
}

sim_windows_256 = np.array(
    simulate_garch_from_windows(train_windows_256, **config)
)

sim_data_256 = {
    "window_size": 256,
    "config": config.copy(),
    "sim_data": sim_windows_256
}

sim_windows_512 = np.array(
    simulate_garch_from_windows(train_windows_512, **config)
)

sim_data_512 = {
    "window_size": 512,
    "config": config.copy(),
    "sim_data": sim_windows_512
}

d:\CodingHenry\research_MBKM\venv\Lib\site-packages\arch\univariate\base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0001672. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
d:\CodingHenry\research_MBKM\venv\Lib\site-packages\arch\univariate\base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 8.439e-05. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)
d:\CodingHenry\research_MBKM\venv\Li

In [26]:
sim_data_128["sim_data"].shape, sim_data_256["sim_data"].shape, sim_data_512["sim_data"].shape,

((18, 128), (9, 256), (4, 512))

In [27]:
SYN_PATH_128 = dirs / "data" / "sim_data_128.joblib"
SYN_PATH_256 = dirs / "data" / "sim_data_256.joblib"
SYN_PATH_512 = dirs / "data" / "sim_data_512.joblib"
joblib.dump(sim_data_128, SYN_PATH_128)
joblib.dump(sim_data_256, SYN_PATH_256)
joblib.dump(sim_data_512, SYN_PATH_512)

['D:\\CodingHenry\\research_MBKM\\data\\sim_data_512.joblib']